In [20]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import when, col
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from google.colab import files
import shutil

# Création de la session Spark
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .getOrCreate()

# Chargement des données
df = spark.read.csv("bank_transactions.csv", header=True, inferSchema=True)

# Renommer toutes les colonnes en minuscules
df = df.select([col(c).alias(c.lower()) for c in df.columns])

# Affichage du schéma
df.printSchema()

# Statistiques descriptives
df.describe().show()

# Conversion des colonnes catégorielles en numériques
indexers = [
    StringIndexer(inputCol=column, outputCol=column + "_index").fit(df)
    for column in ["transactiontype", "location", "channel", "customeroccupation"]
]

# Colonnes de caractéristiques
feature_columns = [
    "transactionamount",
    "customerage",
    "transactionduration",
    "loginattempts",
    "accountbalance",
    "transactiontype_index",
    "location_index",
    "channel_index",
    "customeroccupation_index"
]

# Assemblage des caractéristiques
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Normalisation
scaler = StandardScaler(inputCol="features", outputCol="scaledfeatures")

# Génération de la colonne isfraud
df = df.withColumn("isfraud",
    when((col("transactionamount") > 1000) & (col("transactionduration") < 30), 1)
    .when(col("loginattempts") > 3, 1)
    .when((col("customerage") < 25) & (col("transactionamount") > 500), 1)
    .otherwise(0)
)

# Distribution des classes
df.groupBy("isfraud").count().show()

# Configuration du MLP
layers = [len(feature_columns), 64, 32, 2]

mlp = MultilayerPerceptronClassifier(
    layers=layers,
    featuresCol="scaledfeatures",
    labelCol="isfraud",
    maxIter=100,
    blockSize=128,
    seed=1234
)

# Pipeline
pipeline = Pipeline(stages=indexers + [assembler, scaler, mlp])

# Split train/test
train, test = df.randomSplit([0.7, 0.3], seed=1234)

# Entraînement
model = pipeline.fit(train)

# Prédiction
predictions = model.transform(test)

# Évaluation
evaluator = BinaryClassificationEvaluator(
    labelCol="isfraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)
print(f"Area Under ROC = {auc}")

# Matrice de confusion
predictions.groupBy("isfraud", "prediction").count().show()

# Grid Search
paramGrid = ParamGridBuilder() \
    .addGrid(mlp.maxIter, [50, 100]) \
    .addGrid(mlp.blockSize, [64, 128]) \
    .addGrid(mlp.stepSize, [0.01, 0.1]) \
    .build()

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

cvModel = crossval.fit(train)
bestModel = cvModel.bestModel

# Prédiction finale
bestPredictions = bestModel.transform(test)
bestAuc = evaluator.evaluate(bestPredictions)
print(f"Best Area Under ROC = {bestAuc}")

# Sauvegarde
model_path = "/content/drive/MyDrive/fraud_detection_model"
bestModel.write().overwrite().save(model_path)

shutil.make_archive('fraud_detection_model', 'zip', model_path)
files.download('fraud_detection_model.zip')


root
 |-- transactionid: string (nullable = true)
 |-- accountid: string (nullable = true)
 |-- transactionamount: double (nullable = true)
 |-- transactiondate: timestamp (nullable = true)
 |-- transactiontype: string (nullable = true)
 |-- location: string (nullable = true)
 |-- deviceid: string (nullable = true)
 |-- ip address: string (nullable = true)
 |-- merchantid: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- customerage: integer (nullable = true)
 |-- customeroccupation: string (nullable = true)
 |-- transactionduration: integer (nullable = true)
 |-- loginattempts: integer (nullable = true)
 |-- accountbalance: double (nullable = true)
 |-- previoustransactiondate: timestamp (nullable = true)

+-------+-------------+---------+------------------+---------------+-----------+--------+--------------+----------+-------+------------------+------------------+-------------------+------------------+------------------+
|summary|transactionid|accountid| transacti

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>